## 0. 데이터 전처리 사전 준비

- reviews 폴더에 크롤링된 엑셀 데이터를 저장한다.
- 각 데이터의 주요 컬럼은 다음과 같다.
  - URL: 리뷰가 등록된 가게 URL
  - 가게명: 가게 이름
  - RATING: 별점
  - CONTENT: 리뷰 본문
  - MENU: 주문 메뉴
  - IMAGE_URLS: 리뷰 이미지 URL
  - created_at: 리뷰 생성 시각
  - 지역: 지역

### 목표
- MENU 컬럼에 "리뷰이벤트", "리뷰 이벤트", "리뷰참여" 등의 문구가 포함된 경우 label=1로 설정한다.
- 해당 문구가 없는 경우 label=0으로 설정한다.
- IMAGE_URLS 컬럼을 이용하여 사진 첨부 여부(has_photo)를 생성한다.
- 임베딩을 위한 컬럼 및 원본 컬럼등을 설정한다.

엑셀 파일을 열기 위한 사전 처리
```zsh
uv add openpyxl transformers tokenizers emoji soynlp
```
 - 패키지 필요한 이유
 1. openpyxl	.xlsx 엑셀 파일 읽기
 2. transformers	KcBERT 모델/토크나이저 불러오기
 3. tokenizers	Hugging Face 토크나이저 동작
 4. emoji	이모지 제거/개수 계산
 5. soynlp	ㅋㅋㅋㅋ → ㅋㅋ 반복 문자 정규화

## 1. import
- 라이브러리 및 KCBERT 클린을 위한 임포트

In [ ]:
from pathlib import Path
import re
import os
import pandas as pd
import numpy as np

import emoji
from soynlp.normalizer import repeat_normalize

import torch
from transformers import AutoTokenizer, AutoModel

## 2. 엑셀 파일 로드 함수 설정 및 로드
- xlsx 및 xls를 한 번에 엽니다.
- 0번 행을 제외하고 1번 부터 읽습니다.
- 그리고 다시 인덱스를 0번 부터 갈 수 있도록 초기화합니다.

In [ ]:
def load_excel_files(folder_path, remove_first_row=True):
    folder_path = Path(folder_path)
    excel_files = list(folder_path.rglob("*.xlsx")) + list(folder_path.rglob("*.xls"))

    if len(excel_files) == 0:
        raise FileNotFoundError(f"엑셀 파일을 찾을 수 없습니다: {folder_path}")

    df_list = []

    for file_path in excel_files:
        temp_df = pd.read_excel(file_path)

        if remove_first_row:
            temp_df = temp_df.iloc[1:].reset_index(drop=True)

        temp_df["source_file"] = file_path.name
        df_list.append(temp_df)

    return pd.concat(df_list, ignore_index=True)


df = load_excel_files("reviews", remove_first_row=True)

print("로드 완료:", df.shape)
df.head(3)

## 3. 컬럼 정리
- 한글 컬럼 -> 영어로 변환
- 안쓰는 컬럼 삭제

In [ ]:
df = df.rename(columns={
    "URL": "store_url",
    "가게명": "store_name",
    "NAME": "reviewer_name",
    "RATING": "rating",
    "CONTENT": "review_text",
    "MENU": "menu_name",
    "DATE": "review_date",
    "IMAGE_URLS": "image_urls",
    "OWNER_REPLY": "owner_reply",
    "created_at": "crawled_at",
    "지역": "region"
})

use_cols = [
    "store_url",
    "store_name",
    "rating",
    "review_text",
    "menu_name",
    "image_urls",
    "source_file"
]

df = df[use_cols]
df.head(3)

## 4. 기본 전처리

1. `review_text_raw`: 원문 리뷰 텍스트 보존
2. `menu_name_raw`: 원문 메뉴명 보존
3. 리뷰 텍스트가 비어 있는 행 제거
4. 별점(`rating`)을 숫자형 데이터로 변환

In [ ]:
df["review_text_raw"] = df["review_text"].fillna("").astype(str)
df["menu_name_raw"] = df["menu_name"].fillna("").astype(str)

df = df[df["review_text_raw"].str.strip() != ""].reset_index(drop=True)

df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

df.head(3)

## 5. 리뷰 이벤트 여부 판단 후 레이블화
- 메뉴 기준으로 레이블 생성 후 리뷰 라면 

In [ ]:
event_menu_pattern = re.compile(
    r"(리뷰\s*이벤트|리뷰이벤트|리뷰\s*참여|리뷰참여|리뷰\s*약속|리뷰약속|review\s*event|ri\s*뷰|니\s*뷰|행복\s*음료|500원의\s*행복|아메리카노/매실/아이스티|rㅣ뷰\s*이벤트|rㅣ뷰\s*eㅣ벤트|포토\s*리뷰|리뷰\s*서비스|리뷰\s*할인|\(리뷰\)|리뷰두\s*찜두|100원의\s*행복|백원의\s*행복)",
    flags=re.IGNORECASE
)

def label_review_event_from_menu(menu):
    menu = str(menu)
    return 1 if event_menu_pattern.search(menu) else 0

df["label"] = df["menu_name_raw"].apply(label_review_event_from_menu)
df.head(3)

## 6. KcBERT 입력을 위한 텍스트 정제 및 메타 데이터 생성
- https://github.com/Beomi/KcBERT
- 깃허브 링크 내부의 이모지 관련 체크
- 메타 데이터는 다음과 같다.
    1. 리뷰 텍스트 길이
    2. 이모지 개수
    3. 사진 여부 및 사진 개수


In [ ]:
# KcBERT를 위한 전처리
pattern = re.compile(r'[^ .,?!/@$%~％·∼()\x00-\x7F ㄱ-ㅣ가-힣]+')

url_pattern = re.compile(
    r'https?:\/\/(www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}'
    r'\.[a-zA-Z0-9()]{1,6}\b'
    r'([-a-zA-Z0-9()@:%_\+.~#?&//=]*)'
)

def clean_review_text(text):
    text = str(text)
    text = url_pattern.sub("", text)
    text = emoji.replace_emoji(text, replace="")
    text = pattern.sub(" ", text)
    text = text.strip()
    text = repeat_normalize(text, num_repeats=2)
    return text

df["cleaned_review_text"] = df["review_text_raw"].apply(clean_review_text)

# 메타 데이터 생성
def has_photo(image_urls):
    if pd.isna(image_urls):
        return 0

    image_urls = str(image_urls).strip()

    if image_urls == "" or image_urls in ["[]", "None", "nan", "NaN"]:
        return 0

    return 1


def count_photos(image_urls):
    if pd.isna(image_urls):
        return 0

    image_urls = str(image_urls).strip()

    if image_urls == "" or image_urls in ["[]", "None", "nan", "NaN"]:
        return 0

    return image_urls.count("http")


df["has_photo"] = df["image_urls"].apply(has_photo)
df["photo_count"] = df["image_urls"].apply(count_photos)

def count_emoji(text):
    return len([char for char in str(text) if char in emoji.EMOJI_DATA])

df["text_length"] = df["review_text_raw"].apply(len)
df["emoji_count"] = df["review_text_raw"].apply(count_emoji)

df.head(3)

## 7. 중복제거
- 크롤링되면서 같은 데이터인 경우를 삭제

In [ ]:
df = df.drop_duplicates(
    subset=["store_url", "store_name", "review_text_raw", "menu_name_raw"]
).reset_index(drop=True)

df.head(3)

## 8. 데이터 저장
- 저장
- 작업 폴더 안에 생성됨

In [ ]:
df.to_csv("preprocessed_reviews.csv", index=False, encoding="utf-8-sig")
print(os.path.exists("preprocessed_reviews.csv"))

## 9. 저장된 CSV 테스트

In [ ]:
df_saved = pd.read_csv("preprocessed_reviews.csv", encoding="utf-8-sig")

print("리뷰 개수:", len(df_saved))
print("컬럼 개수:", len(df_saved.columns))
print("데이터 크기:", df_saved.shape)
print("리뷰 이벤트 개수:", (df_saved["label"] == 1).sum())
print("일반 리뷰 개수:", (df_saved["label"] == 0).sum())

df_saved.head(10)

## 10. 분석

In [ ]:
import pandas as pd

# 9번 셀에서 불러온 저장된 CSV 데이터프레임 (이미 로드되어 있다고 가정)
# df_saved = pd.read_csv("preprocessed_reviews.csv", encoding="utf-8-sig")

# 1. 일반 리뷰와 이벤트 리뷰 분리 (label: 0=일반, 1=이벤트)
normal_df = df_saved[df_saved["label"] == 0]
event_df = df_saved[df_saved["label"] == 1]

# 2. 데이터 수 및 전체 데이터 수 계산
normal_count = len(normal_df)
event_count = len(event_df)
total_count = len(df_saved)

# 3. 데이터프레임으로 결과 표 구성
result_data = {
    "항목": [
        "데이터 수",
        "비율",
        "평균 리뷰 길이",
        "평균 이모지 수",
        "평균 사진 수"
    ],
    "일반 리뷰": [
        f"{normal_count:,}",
        f"{(normal_count / total_count * 100):.1f}%",
        f"{normal_df['text_length'].mean():.1f}",
        f"{normal_df['emoji_count'].mean():.1f}",
        f"{normal_df['photo_count'].mean():.1f}"
    ],
    "이벤트 리뷰": [
        f"{event_count:,}",
        f"{(event_count / total_count * 100):.1f}%",
        f"{event_df['text_length'].mean():.1f}",
        f"{event_df['emoji_count'].mean():.1f}",
        f"{event_df['photo_count'].mean():.1f}"
    ]
}

result_table = pd.DataFrame(result_data)

# 결과 출력
print("=== 리뷰 분석 결과 표 ===")
print(result_table.to_string(index=False))